# dataloader-batching composite — cx24: DataLoader batch loop with monotonically increasing step counter

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `dataloader-batching`, `step-counter-increment`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader, TensorDataset

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "dataloader-batching"
DD_ATOM_IDS = ["dataloader-batching", "step-counter-increment"]
DD_SUBTOPICS = ["PyTorch: DataLoader batching", "Trainer: step counter increment"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

A real trainer needs to count **micro-batches across epochs**, not just iterations within one epoch. The composition:

1. **DataLoader batching** — each epoch yields `ceil(N/B)` batches. The same `DataLoader` instance can be iterated multiple times (one outer `for epoch in range(E):`).
2. **Step counter increment** — `step` is incremented ONCE per batch, across ALL epochs. After 3 epochs of 4 batches each, `step == 12`, not `step == 4`.

**Why care.** Logging and LR schedules key off the global step, not the per-epoch step. If you reset `step` at the top of each epoch, your wandb x-axis goes backwards every epoch — the tell-tale visual symptom of this bug.

**Anatomy.**
```python
loader = DataLoader(ds, batch_size=B)         # dataloader-batching.
step = 0
for epoch in range(n_epochs):
    for xb, yb in loader:
        step += 1                              # step-counter-increment, INSIDE batch loop.
        # ... train ...
```

### Composite Exercise — DataLoader batch loop with monotonically increasing step counter

**Atoms exercised together**: `dataloader-batching`, `step-counter-increment`

Implement `cx24_dataloader_step_counter(x_data, y_data, batch_size, n_epochs)`.

Inputs:
- `x_data`, `y_data` — `t.Tensor`s of shape `(N,)` each.
- `batch_size` — int.
- `n_epochs` — int.

Required behaviour:
1. Build a `TensorDataset(x_data, y_data)` and wrap in `DataLoader(ds, batch_size=batch_size, shuffle=False)` (atom: dataloader-batching).
2. Initialise `step = 0` BEFORE the epoch loop (NOT inside).
3. Initialise `step_history = []` — a list to which you append the GLOBAL step number after every batch.
4. For each epoch in `range(n_epochs)`:
   - For each `(xb, yb)` in the loader:
     - Increment `step += 1` FIRST (atom: step-counter-increment).
     - Append `step` to `step_history`.
5. Return `(step, step_history)`.

Test checks:
- `step` is exactly `n_epochs * ceil(N / batch_size)`.
- `step_history == [1, 2, 3, ...]` — monotonically increasing from 1, ONE PER BATCH, across all epochs.
- A buggy 'reset step at top of epoch' would produce `[1, 2, 3, 4, 1, 2, 3, 4, ...]` which the test catches.
- Zero epochs yields `step == 0` and `step_history == []`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx24_dataloader_step_counter(x_data, y_data, batch_size, n_epochs):
    """Iterate a DataLoader for n_epochs; return (final_step, step_history). Step counter is global."""
    raise NotImplementedError

def _test_cx24():
    import math

    # Case A: 2 epochs * 4 batches/epoch = 8 total steps.
    x = t.arange(8.0)   # N=8.
    y = 2.0 * x
    final_step, hist = cx24_dataloader_step_counter(x, y, batch_size=2, n_epochs=2)
    # N=8, B=2 → 4 batches per epoch; 2 epochs → 8 total.
    assert final_step == 8, f'expected final step 8; got {final_step}'
    assert hist == [1, 2, 3, 4, 5, 6, 7, 8], f'step history must be monotonic 1..8; got {hist}'

    # Case B: zero epochs → step stays 0, history is empty.
    fs2, hist2 = cx24_dataloader_step_counter(x, y, batch_size=2, n_epochs=0)
    assert fs2 == 0
    assert hist2 == []

    # Case C: 3 epochs * 3 batches (N=6, B=2) = 9 total steps.
    x3 = t.arange(6.0)
    y3 = t.arange(6.0)
    fs3, hist3 = cx24_dataloader_step_counter(x3, y3, batch_size=2, n_epochs=3)
    assert fs3 == 9, f'3 epochs * 3 batches = 9 steps; got {fs3}'
    assert hist3 == list(range(1, 10)), f'expected [1..9]; got {hist3}'

    # Case D: partial last batch is still ONE batch step.
    x4 = t.arange(7.0)   # N=7, B=3 → batches of 3,3,1 = 3 batches.
    y4 = t.arange(7.0)
    fs4, hist4 = cx24_dataloader_step_counter(x4, y4, batch_size=3, n_epochs=2)
    # 2 epochs * 3 batches = 6 total.
    assert fs4 == 6, f'2 epochs * 3 batches (incl partial) = 6; got {fs4}'
    assert hist4 == [1, 2, 3, 4, 5, 6]

    # Case E: step counter does NOT reset between epochs — explicit guard.
    # A buggy impl that does `step = 0` inside the epoch loop would give [1,2,3,1,2,3].
    # Our correct impl gives [1,2,3,4,5,6]. We assert the correct sequence above. Also confirm
    # that hist[len(epoch1):] starts at len(epoch1)+1, NOT at 1.
    batches_per_epoch = math.ceil(7 / 3)   # = 3.
    first_step_of_epoch2 = hist4[batches_per_epoch]
    assert first_step_of_epoch2 == batches_per_epoch + 1, (
        f'first step of epoch 2 should be {batches_per_epoch + 1} (no reset); '
        f'got {first_step_of_epoch2} — looks like the step counter was reset at top of epoch'
    )
    _dd_passed.add('cx24')

_test_cx24()

<details><summary>Show solution — cx24</summary>

```python
def cx24_dataloader_step_counter(x_data, y_data, batch_size, n_epochs):
    # Atom A (dataloader-batching).
    ds = TensorDataset(x_data, y_data)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    # Step counter lives OUTSIDE the epoch loop — global across all epochs.
    step = 0
    step_history = []
    for _epoch in range(n_epochs):
        for _xb, _yb in loader:
            # Atom B (step-counter-increment): bump BEFORE recording.
            step += 1
            step_history.append(step)
    return step, step_history
```

The trap this exercise catches is putting `step = 0` *inside* the epoch loop. It looks innocuous because it 'works' within a single epoch, but the wandb dashboard, the LR scheduler, and the eval-every-K-steps callback all silently misbehave. The fix is just moving the assignment one indent level up — but it requires understanding the contract (global step) over the local pattern (per-epoch loop).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx24'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx24',
        'subtopics': ["PyTorch: DataLoader batching", "Trainer: step counter increment"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()